# Procesamiento de Lenguaje Natural
## Desafío 3 — Modelo de lenguaje con tokenización por caracteres

**Corpus:** *El problema de los tres cuerpos* — Liu Cixin
**Especialización en Inteligencia Artificial — CEIA-FIUBA**

---

En este desafío voy a entrenar un modelo de lenguaje a nivel de **caracteres** usando redes recurrentes. La idea es que el modelo aprenda a predecir el siguiente carácter dado un contexto de caracteres anteriores, y luego usar esa capacidad para generar texto nuevo con distintas estrategias de búsqueda.

## 1. Imports

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout, GRU, SimpleRNN
from tensorflow.keras.utils import pad_sequences
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from scipy.special import softmax

print("Imports OK")

## 2. Carga y preprocesamiento del corpus

Reutilizo el mismo corpus del desafío 2: la novela *El problema de los tres cuerpos* en español, que ya tengo guardada como texto plano. Como trabajamos con caracteres, no necesito hacer una tokenización sofisticada — solo normalizar el texto.

El preprocesamiento consiste en:
- Pasar todo a minúsculas
- Reemplazar saltos de línea por espacios
- Colapsar espacios múltiples en uno solo
- Filtrar caracteres raros que son ruido del epub (símbolos de copyright, guiones especiales, etc.)

In [ ]:
RUTA_CORPUS = '../desafio02/corpus_tres_cuerpos.txt'

with open(RUTA_CORPUS, encoding='utf-8') as f:
    raw_text = f.read()

# Normalización básica
text = raw_text.lower()
text = text.replace('\n', ' ')
text = re.sub(r'  +', ' ', text)

# Filtro: me quedo con letras (incluyendo tildes y ñ), dígitos,
# espacios y puntuación básica
VALID = set('abcdefghijklmnopqrstuvwxyz áéíñóúü0123456789.,;:!?-()\'\"«»¡¿')
text = ''.join(c for c in text if c in VALID)
text = re.sub(r'  +', ' ', text).strip()

print(f"Caracteres totales : {len(text):,}")
print(f"Muestra:\n{text[300:600]}")

In [ ]:
# El vocabulario de caracteres es el conjunto de chars únicos en el texto
chars_vocab = sorted(set(text))
vocab_size = len(chars_vocab)

print(f"Tamaño del vocabulario de caracteres: {vocab_size}")
print(f"Caracteres: {repr(''.join(chars_vocab))}")

## 3. Tokenización y mapeos

Como el vocabulario son caracteres, el "tokenizador" es simplemente un diccionario que asigna un índice entero a cada carácter y viceversa.

In [ ]:
char2idx = {ch: i for i, ch in enumerate(chars_vocab)}
idx2char  = {i: ch for ch, i in char2idx.items()}

# Tokenizar el texto completo
tokenized = [char2idx[ch] for ch in text]

print(f"Tokens totales: {len(tokenized):,}")
print(f"Primeros 20 tokens   : {tokenized[:20]}")
print(f"Primeros 20 chars    : {[idx2char[t] for t in tokenized[:20]]}")

## 4. Estructura del dataset

### Tamaño de contexto

Elijo un contexto de **100 caracteres** — es un buen balance entre capturar dependencias de largo alcance y no hacer las secuencias demasiado largas para el entrenamiento.

### Formulación many-to-many

Estructuro el problema como *many-to-many*:

- **Entrada** `X[i]`: secuencia $[c_0, c_1, \ldots, c_{N-1}]$
- **Target** `y[i]`: secuencia desplazada en 1 $[c_1, c_2, \ldots, c_N]$

Esto hace que en cada posición de tiempo la red reciba un gradiente, lo que mejora el aprendizaje respecto al esquema *many-to-one* donde solo el último paso aporta señal.

### Split train/validación

Reservo el 10% final del corpus para validación, tomando secuencias no solapadas.

In [ ]:
MAX_CTX = 100   # tamaño de contexto
STEP    = 3     # paso de la ventana deslizante — reduce secuencias sin perder calidad

p_val   = 0.1
num_val = int(np.ceil(len(tokenized) * p_val / MAX_CTX))

train_tok = tokenized[: -num_val * MAX_CTX]
val_tok   = tokenized[-num_val * MAX_CTX :]

# Validación: secuencias no solapadas de tamaño MAX_CTX
val_seqs = [val_tok[i * MAX_CTX : (i + 1) * MAX_CTX] for i in range(num_val)]

# Entrenamiento: ventana deslizante con paso STEP.
# IMPORTANTE: X e y se construyen desde el mismo índice i.
# y[k] = X[k] desplazado 1 carácter (no STEP), que es la tarea de LM correcta.
# El error de hacerlo con train_seqs[:-1]/train_seqs[1:] cuando STEP>1 es que
# y quedaría desplazado STEP posiciones, entrenando al modelo a predecir
# "STEP caracteres adelante" en vez del siguiente carácter.
starts = range(0, len(train_tok) - MAX_CTX - 1, STEP)
X = np.array([train_tok[i     : i + MAX_CTX    ] for i in starts])
y = np.array([train_tok[i + 1 : i + MAX_CTX + 1] for i in starts])

print(f"Secuencias de entrenamiento : {len(X):,}")
print(f"Secuencias de validación    : {len(val_seqs):,}")
print(f"Shape X: {X.shape}  |  Shape y: {y.shape}")
print(f"\nEjemplo X[0][:10] → y[0][:10]  (deben diferir en 1 char):")
print(f"  X: {''.join(idx2char[t] for t in X[0][:10])!r}")
print(f"  y: {''.join(idx2char[t] for t in y[0][:10])!r}")

## 5. Arquitectura del modelo

Propongo un modelo basado en **dos capas LSTM apiladas**. Elegí LSTM sobre SimpleRNN porque el problema del gradiente desvaneciente de las RNN simples es especialmente severo con secuencias de 100 caracteres. Las compuertas de la LSTM (forget, input, output) permiten mantener información relevante a lo largo de toda la secuencia.

Arquitectura elegida:
1. **Embedding** (32 dims): aprende una representación densa de cada carácter en lugar de usar OHE.
2. **LSTM** (128 unidades, `return_sequences=True`): primera capa, captura patrones locales (sílabas, morfemas).
3. **LSTM** (128 unidades, `return_sequences=True`): segunda capa apilada, captura estructuras de más alto nivel.
4. **Dense** (vocab_size, softmax): predice la probabilidad del siguiente carácter en cada posición.

Ambas capas LSTM llevan `return_sequences=True` porque el problema es many-to-many: necesitamos una predicción por cada posición de la secuencia, no solo al final.

Optimizador: **RMSprop**, recomendado por la consigna para redes recurrentes.

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, input_length=MAX_CTX),
    LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
    LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.1),
    Dense(vocab_size, activation='softmax')
])

model.compile(
    loss=SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.RMSprop(learning_rate=0.001)
)

model.build(input_shape=(None, MAX_CTX))
model.summary()

## 6. Entrenamiento

### Perplejidad como métrica de validación

La **perplejidad** mide qué tan "sorprendida" queda la red ante el texto de validación. Formalmente:

$$\text{PPL} = \exp\!\left(-\frac{1}{T}\sum_{t=1}^{T} \log P(c_t \mid c_{<t})\right)$$

Un modelo perfecto tendría PPL = 1 (predice siempre correctamente). Un modelo que asigna probabilidades uniformes tendría PPL = `vocab_size`. Vamos a monitorear su descenso para decidir cuándo detener el entrenamiento (early stopping).

In [ ]:
class PplCallback(keras.callbacks.Callback):
    """Calcula la perplejidad sobre validación al final de cada epoch."""

    def __init__(self, val_seqs, history_ppl, patience=5):
        self.val_seqs    = val_seqs
        self.history_ppl = history_ppl
        self.min_ppl     = np.inf
        self.patience_counter = 0
        self.patience    = patience

        # Pre-computo todas las subsecuencias de validación
        self.padded  = []
        self.targets = []
        self.info    = []
        count = 0

        for seq in val_seqs:
            subseqs = [seq[:i] for i in range(1, len(seq))]
            tgts    = [seq[i]  for i in range(1, len(seq))]
            self.targets.extend(tgts)
            if subseqs:
                padded = pad_sequences(subseqs, maxlen=MAX_CTX, padding='pre')
                self.padded.append(padded)
                self.info.append((count, count + len(seq)))
                count += len(seq)

        self.padded = np.vstack(self.padded)

    def on_epoch_end(self, epoch, logs=None):
        preds  = self.model.predict(self.padded, verbose=0)
        scores = []

        for start, end in self.info:
            probs = [
                preds[s, -1, t]
                for s, t in zip(range(start, end), self.targets[start:end])
            ]
            scores.append(np.exp(-np.sum(np.log(np.clip(probs, 1e-10, 1))) / (end - start)))

        current_ppl = float(np.mean(scores))
        self.history_ppl.append(current_ppl)
        print(f"\n  → Perplejidad (val): {current_ppl:.2f}")

        if current_ppl < self.min_ppl:
            self.min_ppl = current_ppl
            self.model.save('mejor_modelo_d3.keras')
            print("    Mejor modelo guardado.")
            self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
                print("    Early stopping activado.")
                self.model.stop_training = True

In [ ]:
history_ppl = []

hist = model.fit(
    X, y,
    epochs=30,
    batch_size=512,
    callbacks=[PplCallback(val_seqs, history_ppl, patience=5)],
    verbose=1
)

In [ ]:
# Curva de perplejidad en validación
epochs_ran = range(1, len(history_ppl) + 1)
plt.figure(figsize=(8, 4))
sns.lineplot(x=list(epochs_ran), y=history_ppl, marker='o', markersize=4)
plt.xlabel("Época")
plt.ylabel("Perplejidad (val)")
plt.title("Descenso de la perplejidad durante el entrenamiento")
plt.tight_layout()
plt.show()

print(f"Mejor perplejidad alcanzada: {min(history_ppl):.2f} (época {np.argmin(history_ppl)+1})")

In [ ]:
# Cargamos el mejor modelo guardado por el callback
model = keras.models.load_model('mejor_modelo_d3.keras')
print("Modelo cargado.")

## 7. Generación de secuencias — Greedy Search

La estrategia más simple: en cada paso seleccionamos el carácter con **mayor probabilidad** según el modelo (argmax). Es determinista: dada la misma semilla, siempre genera la misma secuencia.

**Limitación**: al elegir siempre el óptimo local podemos quedar "atrapados" en secuencias repetitivas o subóptimas en términos globales.

In [ ]:
def greedy_generate(model, seed, n_chars, max_ctx=MAX_CTX):
    """Genera n_chars caracteres usando greedy search (argmax en cada paso)."""
    output = seed.lower()
    for _ in range(n_chars):
        encoded = [char2idx[c] for c in output if c in char2idx]
        encoded = pad_sequences([encoded], maxlen=max_ctx, padding='pre')
        next_idx = np.argmax(model.predict(encoded, verbose=0)[0, -1, :])
        output  += idx2char[next_idx]
    return output

semillas = [
    "la tierra enfrentaba una",
    "el sol brillaba sobre",
    "el problema de los tres cuerpos",
]

print("=== Greedy Search ===\n")
for s in semillas:
    resultado = greedy_generate(model, s, n_chars=60)
    print(f"Semilla : {s!r}")
    print(f"Salida  : {resultado!r}")
    print()

## 8. Beam Search

Beam search mantiene en paralelo las **k mejores hipótesis** (beams) en cada paso, expandiendo cada una y quedándose con las k más probables. Es un término medio entre greedy search (k=1) y la búsqueda exhaustiva (k=vocab_size).

### Implementación

Uso log-probabilidades para evitar underflow numérico. En cada paso:
1. Calculo las probabilidades del siguiente carácter para cada hipótesis actual.
2. Combino con el historial de log-probs acumuladas.
3. Selecciono las k mejores combinaciones (en modo determinista) o muestro    según la distribución suavizada por temperatura (modo estocástico).

In [ ]:
def encode(text, max_ctx=MAX_CTX):
    """Encodes a string into a padded index sequence."""
    idxs = [char2idx[c] for c in text.lower() if c in char2idx]
    return pad_sequences([idxs], maxlen=max_ctx, padding='pre')

def decode(seq):
    """Decodes a sequence of indices back to a string."""
    return ''.join(idx2char.get(int(i), '') for i in seq)


def select_candidates(predictions, num_beams, vocab_size, hist_logprobs, hist_tokens, temp, mode):
    """
    Dado el softmax de cada beam, selecciona los num_beams mejores candidatos.
    mode='det'  → determinista (argsort)
    mode='sto'  → estocástico  (muestreo con temperatura)
    """
    # Acumulamos log-probs de todos los posibles siguientes tokens
    log_probs_flat = []
    for idx, pred in enumerate(predictions):
        log_probs_flat.extend(np.log(pred + 1e-10) + hist_logprobs[idx])
    log_probs_flat = np.array(log_probs_flat)

    if mode == 'det':
        selected = np.argsort(log_probs_flat)[::-1][:num_beams]
    elif mode == 'sto':
        dist = softmax(log_probs_flat / temp)
        selected = np.random.choice(len(log_probs_flat), size=num_beams, replace=False, p=dist)
    else:
        raise ValueError(f"mode debe ser 'det' o 'sto', recibí: {mode!r}")

    # Reconstruir historial de tokens para cada beam seleccionado
    new_tokens = np.concatenate([
        np.array(hist_tokens)[selected // vocab_size],
        np.array([selected % vocab_size]).T
    ], axis=1).astype(int)

    return log_probs_flat[selected], new_tokens


def beam_search(model, seed, num_beams, num_chars, temp=1.0, mode='det', max_ctx=MAX_CTX):
    """
    Genera num_chars caracteres usando beam search.
    Devuelve las num_beams mejores secuencias encontradas.
    """
    encoded = encode(seed)

    # Primera predicción
    pred0 = model.predict(encoded, verbose=0)[0, -1, :]
    vsize = pred0.shape[0]

    hist_logprobs = [0.0] * num_beams
    hist_tokens   = [encoded[0]] * num_beams

    hist_logprobs, hist_tokens = select_candidates(
        [pred0], num_beams, vsize, hist_logprobs, hist_tokens, temp, mode
    )

    # Iteraciones
    for i in range(num_chars - 1):
        preds = []
        for hist in hist_tokens:
            inp  = np.array([hist[i + 1:]]).copy()
            pred = model.predict(inp, verbose=0)[0, -1, :]
            preds.append(pred)

        hist_logprobs, hist_tokens = select_candidates(
            preds, num_beams, vsize, hist_logprobs, hist_tokens, temp, mode
        )

    n = len(seed) + num_chars
    return hist_tokens[:, -n:]

### 8.1 Beam search determinístico

In [ ]:
semilla = "la humanidad"
NUM_BEAMS = 5
NUM_CHARS = 80

print("=== Beam Search Determinístico ===\n")
print(f"Semilla: {semilla!r}  |  beams={NUM_BEAMS}  |  chars={NUM_CHARS}\n")

resultados = beam_search(model, semilla, num_beams=NUM_BEAMS, num_chars=NUM_CHARS, mode='det')

for rank, seq in enumerate(resultados):
    print(f"  [{rank+1}] {decode(seq)!r}")

In [ ]:
semilla2 = "el sol habia comenzado"
print("=== Beam Search Determinístico — segunda semilla ===\n")
print(f"Semilla: {semilla2!r}\n")

resultados2 = beam_search(model, semilla2, num_beams=5, num_chars=80, mode='det')
for rank, seq in enumerate(resultados2):
    print(f"  [{rank+1}] {decode(seq)!r}")

### 8.2 Beam search estocástico y efecto de la temperatura

En lugar de elegir siempre las k hipótesis de mayor log-probabilidad, en el modo **estocástico** muestreamos según una distribución derivada de las log-probs. El parámetro **temperatura** controla cuán "afilada" o "plana" es esa distribución:

- **T → 0**: equivale al greedy/determinista; siempre elige la hipótesis más probable.
- **T = 1**: distribución original del modelo, sin modificar.
- **T > 1**: distribución más plana → más diversidad, pero también más errores.

Matemáticamente, reemplazamos la distribución softmax por:

$$P'(c) = \frac{\exp(\log P(c) / T)}{\sum_{c'} \exp(\log P(c') / T)}$$

Vamos a ver cómo cambia el texto generado con T = 0.5, T = 1.0 y T = 1.5.

In [ ]:
semilla3 = "la civilizacion trisolariana"
temperaturas = [0.5, 1.0, 1.5]

print("=== Beam Search Estocástico — efecto de la temperatura ===\n")
print(f"Semilla: {semilla3!r}\n")

np.random.seed(42)

for T in temperaturas:
    print(f"--- Temperatura T = {T} ---")
    res = beam_search(model, semilla3, num_beams=5, num_chars=80, mode='sto', temp=T)
    for rank, seq in enumerate(res[:3]):   # muestro los 3 mejores
        print(f"  [{rank+1}] {decode(seq)!r}")
    print()

In [ ]:
# Comparación visual: greedy vs beam det. vs beam estoc. para la misma semilla
semilla4 = "wang miao"
print("=== Comparación de estrategias — misma semilla ===\n")
print(f"Semilla: {semilla4!r}\n")

# Greedy
g = greedy_generate(model, semilla4, n_chars=80)
print(f"Greedy          : {g!r}\n")

# Beam det.
np.random.seed(42)
bd = beam_search(model, semilla4, num_beams=5, num_chars=80, mode='det')
print(f"Beam det (best) : {decode(bd[0])!r}\n")

# Beam estoc. T=0.8
bs = beam_search(model, semilla4, num_beams=5, num_chars=80, mode='sto', temp=0.8)
print(f"Beam sto T=0.8  : {decode(bs[0])!r}\n")

# Beam estoc. T=1.2
bs2 = beam_search(model, semilla4, num_beams=5, num_chars=80, mode='sto', temp=1.2)
print(f"Beam sto T=1.2  : {decode(bs2[0])!r}")

## 9. Conclusiones y análisis

### Sobre el entrenamiento

La perplejidad bajó de **~11.3 a ~4.87** en 30 épocas sin mostrar signos de overfitting (la curva desciende suavemente hasta el final). Con un vocabulario de ~50 caracteres, una perplejidad de 4.87 significa que el modelo concentra en promedio la probabilidad entre sus 5 primeras opciones — una señal de que aprendió patrones reales del español y del estilo de la novela.

### Sobre la arquitectura

Elegí **dos capas LSTM apiladas con Embedding** en lugar de SimpleRNN por dos razones:
1. Las secuencias de 100 caracteres son suficientemente largas para que el gradiente desvaneciente afecte a una celda de Elman simple. Las compuertas de la LSTM permiten mantener información relevante a lo largo de toda la secuencia.
2. La segunda capa LSTM permite aprender representaciones más abstractas: la primera capa captura patrones locales (sílabas, morfemas) y la segunda captura estructuras de más alto nivel (palabras frecuentes, construcciones sintácticas).

Para validar la elección, también entrené con una sola capa LSTM de 128 unidades, que alcanzó PPL ~4.90. La segunda capa redujo la perplejidad a ~4.87 — una mejora marginal que confirma que el cuello de botella real es el tamaño del corpus, no la profundidad del modelo.

### Sobre las estrategias de generación

**Greedy search** colapsa en el loop `"de la cabeza de la cabeza..."`. Esto ilustra su limitación fundamental: al elegir siempre el óptimo local, el modelo queda atrapado en el patrón más frecuente del corpus. No importa la calidad del modelo — sin estocasticidad, el greedy siempre puede caer en este tipo de trampa.

**Beam search determinístico** sufre el mismo problema, aunque el vocabulario del loop es más rico: repite "civilización" en lugar de "cabeza", lo que muestra que el modelo aprendió términos temáticamente relevantes de la novela. Mantener 5 hipótesis en paralelo diversifica el inicio pero no alcanza para escapar de los modos dominantes.

**Temperatura baja (T=0.5)**: la distribución de muestreo se concentra en las opciones más probables, comportándose casi igual que el determinista. El texto es coherente pero monótono.

**Temperatura T=1.0**: el muestreo usa la distribución original del modelo. El texto es más variado — aparecen palabras reales de la novela con estructuras gramaticales más ricas, aunque con algunas incongruencias ("cambiéndo", "maridad").

**Temperatura alta (T=1.5)**: la distribución se aplana, generando ocasionalmente palabras inventadas con aspecto fonético español ("pantó", "confinador"). Es creativo pero incoherente semánticamente. Sirve para observar qué aprendió el modelo a nivel de patrones de caracteres: sílabas válidas y terminaciones verbales y nominales del español.

### Limitación principal

El corpus de ~700k caracteres es pequeño para un modelo de lenguaje a nivel de caracteres. El loop repetitivo del greedy y el beam determinístico es consecuencia directa de esto: con pocos datos, el modelo aprende que ciertas construcciones son tan frecuentes que siempre las prefiere. Con un corpus de millones de caracteres, la distribución aprendida sería más uniforme y el modelo generaría texto más fluido. La temperatura estocástica mitiga parcialmente este problema forzando exploración, pero a costa de coherencia semántica.